## Outils pour la manipulation d'images et librairies.


In [ ]:
import PIL
from PIL import Image
import numpy as np
import scipy as sp
import os
from math import log10, sqrt

def load(filename):
    toLoad= Image.open(filename)
    return np.asarray(toLoad)


def psnr(original, compressed):
    mse = np.mean((original - compressed) ** 2)
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse))
    return psnr

def dct2(a):
    return sp.fft.dct( sp.fft.dct( a, axis=0, norm='ortho' ), axis=1, norm='ortho' )

def idct2(a):
    return sp.fft.idct( sp.fft.idct( a, axis=0 , norm='ortho'), axis=1 , norm='ortho')

## Normalisation de l'image (YCbCr et padding)

## Découpage en blocs et compression

In [ ]:
def ycbcr(mat):
    """Fonction qui transforme une matrice donnée en YCbCR"""
    new_mat = np.empty((mat.shape), dtype = np.uint8)
    new_mat[:,:,0] = 0.299*mat[:,:,0] + 0.587*mat[:,:,1] + 0.114*mat[:,:,2]
    new_mat[:,:,1] = -0.1687*mat[:,:,0] - 0.3313*mat[:,:,1] + 0.5*mat[:,:,2] + 128
    new_mat[:,:,2] = 0.5*mat[:,:,0] - 0.4187*mat[:,:,1] - 0.0813*mat[:,:,2] + 128
    return new_mat


def rgb(mat):
    """Fonction qui transforme une matrice donnée en RGB"""
    mat = np.float64(mat.copy())
    new_mat = np.empty((mat.shape), dtype = np.uint8)       
    new_mat[:,:,0] = np.clip(mat[:,:,0] + 1.402*(mat[:,:,2] - 128),0,255)
    new_mat[:,:,1] = np.clip(mat[:,:,0] - 0.34414*(mat[:,:,1] - 128) - 0.71414*(mat[:,:,2] - 128),0,255)
    new_mat[:,:,2] = np.clip(mat[:,:,0] + 1.772*(mat[:,:,1] - 128),0,255)
    return new_mat


def pad8(val):
    """Fonction qui permet de calculer le padding nécessaire"""
    return val+(8 - val%8)%8


def padding(image):
    """Fonction qui créer le padding sur l'image"""
    new_image = np.zeros((pad8(image.shape[0]), pad8(image.shape[1]),3),dtype = np.uint8)
    new_image[:image.shape[0],:image.shape[1]] = image
    return new_image


def depadding(image,hauteur,largeur):
    """Fonction qui retire le padding créer précédemment"""
    depadimage = image[:hauteur,:largeur]  # Les dimensions de l'image actuel sont égales aux dimensions de l'image de base.
    return depadimage


def sous_echantillonnage(mat):
    """Fonction qui utilse la méthode de sous echantillonage 4 : 2 : 2"""
    mat = np.float64(mat.copy())  # Copie de la matrice en float.
    matY = mat[:,:,0].copy()  # Copie des valeurs de Y dans une matrice matY.
    matCb = np.empty((mat.shape[0],mat.shape[1]//2),dtype=np.float64)  # Création d'une matrice matCb deux fois plus petite que le canal Cb de la matrice de base.
    matCr = np.empty((mat.shape[0],mat.shape[1]//2),dtype=np.float64)  # Création d'une matrice matCr deux fois plus petite que le canal Cr de la matrice de base.
    for i in range(matCb.shape[0]):  # Boucle qui parcourt la matrice matCb en hauteur.
        for j in range(matCb.shape[1]):  # Boucle qui parcourt la matrice matCb en largeur.
            matCb[i,j] = (mat[i,j*2,1] + mat[i,(j*2)+1,1]) // 2  # Moyenne de la valeur de Cb et de Cb + 1.
            matCr[i,j] = (mat[i,j*2,2] + mat[i,(j*2)+1,2]) // 2  # Moyenne de la valeur de Cr et de Cr + 1.
    return(matY,matCb,matCr)  # On return un tuple contenant les différents canaux.


def de_echantillonnage(tuple):
    """Fonction qui prend une image sous échantillonée et qui la remet dans sa taille d'origine"""
    matY = np.uint8(tuple[0].copy())  # matY est égal au tuple[0] c'est à dire Y.
    matCb = np.uint8(tuple[1].copy())  # matCb est égal au tuple[1] c'est à dire Cb.
    matCr = np.uint8(tuple[2].copy())  # matCr est égal au tuple[2] c'est à dire Cr.
    mat = np.empty((matY.shape[0],matY.shape[1],3), dtype=np.uint8)  # Création d'une matrice de la taille de l'image de base.
    mat[:,:,0] = matY.copy()  # On met une copie de Y dans notre matrice.
    for i in range(mat.shape[0]):  # Boucle qui parcourt la matrice en hauteur.
        for j in range(mat.shape[1]):  # Boucle qui parcourt la matrice en largeur.
            mat[i, j,1] = matCb[i,j//2]  # On ajoute à notre matrice la valeur de Cb.
            mat[i, j,2] = matCr[i,j//2]  # On ajoute à notre matrice la valeur de Cr.
    return mat  # On return une matrice correspondant à l'image de taille de base.


def decouper_en_blocs(mat):
    """Fonction qui découpe en 8x8 une image donnée"""
    blocs = []
    h = mat.shape[0]  # h est égal à la hauteur de l'image.
    l = mat.shape[1]  # l est égal à la largeur de l'image.
    nb_blocs_h = h // 8  # Permet de calculer le nombre de blocs dans l'image.
    nb_blocs_l = l // 8
    for i in range(nb_blocs_h):  # On parcourt l'image bloc par bloc.
        for j in range(nb_blocs_l):
            bloc = mat[i*8:(i+1)*8, j*8:(j+1)*8]  # Bloc est égal à un bloc de 8x8.
            blocs.append(bloc)  # On ajoute à blocs chaque bloc.
    return np.array(blocs)  # On return la liste de blocs sous forme de array numpy pour que la lecture soit plus claire.


def regrouper_blocs(blocs, hauteur, largeur):
    """Fonction qui prends les blocs découpée et les regroupe pour reformer l'image de base"""
    image_regroupee = np.zeros((hauteur, largeur, 3), dtype=np.uint8)  # On créer une matrice de la taille de l'image de base.
    nb_blocs_h = hauteur // 8  # On retrouve la taille des blocs.
    nb_blocs_l = largeur // 8
    for i in range(nb_blocs_h):  # On parcourt chaque blocs.
        for j in range(nb_blocs_l):
            bloc = blocs[i * nb_blocs_l + j]  # Bloc est égal à chaque bloc contenu dans blocs.
            image_regroupee[i * 8:(i + 1) * 8, j * 8:(j + 1) * 8] = bloc  # On regroupe les blocs en liste.
    return image_regroupee  # On return l'image non séparée en bloc.


def cos_discret(lbloc):
    """Fonction qui transforme la liste de bloc en fréquence"""
    transfo = []  # Création d'une liste vide.
    for i in range(len(lbloc)):  # On parcourt chaque blocs de notre image.
        transfo.append(dct2(lbloc[i]))  # On applique la transformation à chaque bloc de notre image.
    listetransfo = np.array(transfo)  # Création de listetransfo sous forme de np.array qui possède toutes les fréquences de nos bloc.
    return listetransfo  # On return listetransfo qui possède toutes les fréquences en blocs.


def de_cos_discret(freqbloc):
    """Fonction qui retire les fréquences d'une liste de blocs"""
    detransfo = []  # Création d'une liste vide
    for i in range(len(freqbloc)):  # On parcourt chaque blocs de notre image.
        detransfo.append(idct2(freqbloc[i]))  # On applique la détransformation à chaque bloc de notre image.
    listedetransfo = np.array(detransfo)  # Création de listedetransfo sous forme de np.array qui possède la valeur de tous les blocs, sans fréquences.
    return listedetransfo  # On return listetransfo qui possède la valeur de tous nos blocs.


def mode0(blocs):
    """Fonction qui lance le mode0, c'est à dire la transformation par cosinus discret"""
    return cos_discret(blocs)  # On return le résultat de notre fonction cos_discret, c'est à dire les fréquences de chaque blocs.


def mode1(blocs, seuil):
    """Fonction qui lance le mode1, c'est à dire la transformation par cosinus discret avec un seuil donné"""
    transfo = mode0(blocs)  # On créer transfo qui possède les blocs sous forme de fréquences.
    for i in range(len(transfo)):  # On parcourt notre liste en hauteur.
        for j in range(len(transfo[i])):  # On parcourt notre liste en largeur.
            for k in range(len(transfo[i][j])):  # On parcourt les différentes lignes de notre blocs.
                for l in range(len(transfo[i][j][k])):  # On parcourt les valeurs [Y,Cb,Cr] de notre liste.
                    if transfo[i][j][k][l] < seuil:  # Si notre valeur est en dessous du seuil.
                        transfo[i][j][k][l] = 0  # On passe cette valeur à 0.
    print(transfo)
    return transfo  # On return transfo avec toutes les valeurs en dessous du seuil à 0.


def mode2(blocs,seuil):
    """Fonction qui lance le mode2, c'est à dire la transformation par cosinus discret avec un seuil donné d'une image sous echantillonée"""
    return mode1(blocs,seuil)  # On return toutes les valeurs avec notre seuil appliqué.


def explosion(mat,seuil):
    """Fonction qui lance tous les modes en même temps"""
    matycbcr = ycbcr(padding(rgb(mat)))  # matycbcr est égal au padding de notre image en RGB repassée ensuite en YCbCr.
    return mode0(decouper_en_blocs(matycbcr)), mode1(decouper_en_blocs(matycbcr),seuil) , mode2(decouper_en_blocs(de_echantillonnage(sous_echantillonnage(matycbcr))),seuil)  # On return mode0, mode1, mode2.

## Écriture dans un fichier

In [ ]:
def ecriture(mat,nbr):
    """Fonction qui écrit dans un fichier SJPG"""
    modecomp = str(input("RLE ou NORLE"))  # On choisit le mode de compression.
    if modecomp == "RLE":  # Vérification du mode.
        modecomp = True
    elif modecomp == "NORLE":
        modecomp = False
    else:  # Si on n'a pas écrit le bon mode.
        return 'Rentrez "RLE" ou "NORLE"'
    if nbr == 2:  # On crée la matrice si c'est le mode 2.
        matbloc = decouper_en_blocs(de_echantillonnage(sous_echantillonnage(padding(ycbcr(mat)))))
    else:  # On crée la matrice si c'est le mode 0 ou mode 1.
        matbloc = decouper_en_blocs(padding(ycbcr(mat)))
    
    if nbr==0:  # On crée les blocs compréssés et mode 0.
        blocs_compresses = mode0(matbloc)
    elif nbr==1:  # On crée les blocs compréssés avec le seuil et mode 1.
        seuil=int(input("Seuil : "))
        blocs_compresses = mode1(matbloc,seuil)
    elif nbr==2:  # On crée les blocs compréssés avec le seuil et mode 2 .
        seuil=int(input("Seuil : "))
        blocs_compresses = mode2(matbloc,seuil)
    else:  # Si on n'a pas écrit la bonne valeur.
        return "Choissiez une bonne valeur entre 0, 1 et 2 pour le mode"
    f = open("fichier SJPG","w")  # Création / Ouverture du fichier dans lequel on va écrire.
    f.write("SJPG\n")  # Première ligne où on indique le type SJPG.
    f.write(str(mat.shape[0])+" "+str(mat.shape[1])+"\n")  # Deuxième ligne où on indique la taille de l'image.
    f.write("mode "+str(nbr)+"\n")  # Troisième ligne on indique le mode utilisé.
    if modecomp == False:  # Quatrième ligne on indique le type de compression.
        f.write("NORLE\n")
    else:
        f.write("RLE\n")
    texte = encodage(blocs_compresses,modecomp)  # Création du texte grâce à la fonction encodage.
    f.write(str(texte))  # On écrit le texte reçu.
    f.close()  # On ferme le fichier.
    message = "Le fichier 'fichier SJPG' à bien été crée, en mode" + str(nbr)
    if modecomp:
        message += " en RLE"
    else:
        message += " sans RLE"
    if nbr == 1 or nbr == 2:
        message += " avec un seuil de " +str(seuil)+"."
    else:
        message += "."
    return message


def encodage(blocs_compresses, modecomp):
    """Fonction qui encode en RLE ou NORLE"""
    text = ""
    for i in range(len(blocs_compresses)):  # On parcourt chaques blocs.
        # Boucle Y.
        lignebloc = ""  # On crée la ligne à écrire.
        nbr_zeros = 0  # On crée la variable du nombres de zeros.
        for j in range(len(blocs_compresses[i])):  # On parcourt la taille de chaque blocs.
            for k in range(len(blocs_compresses[i][j])):  # On parcourt chaque pixels.
                if modecomp:  # Si le mode de compression est RLE.
                    if int(blocs_compresses[i][j][k][0]) == 0:  # Si c'est un zero on incrémente le nombre de zeros.
                        nbr_zeros += 1
                    else:
                        if nbr_zeros > 0:  # Si le nombre actuel n'est pas un zero et que le nombre de zeros est positif, on applique le RLE.
                            lignebloc += "#" + str(nbr_zeros) + " "
                            nbr_zeros = 0  # On met le nombre de zeros à zero.
                        lignebloc += str(int(blocs_compresses[i][j][k][0])) + " "  # On ajoute la valeur actuelle à la ligne.
                else:
                   lignebloc += str(int(blocs_compresses[i][j][k][0])) + " "  # Si le mode est NORLE on ajoute chaque valeur une par une.
        # Ajouter le nombre de zéros à la fin du bloc.
        if nbr_zeros > 0:
            lignebloc += "#" + str(nbr_zeros) + " "  # On ajoute le nombre de zeros si c'est la fin du bloc.
        text += lignebloc.strip() + "\n"  # On écrit la ligne dans le texte.
        # Boucle Cb.
        lignebloc = ""  # C'est les mêmes commentaires que pour le Y au dessus.
        nbr_zeros = 0
        for l in range(len(blocs_compresses[i])):
            for m in range(len(blocs_compresses[i][j])):
                if modecomp:
                    if int(blocs_compresses[i][l][m][1]) == 0:
                        nbr_zeros += 1
                    else:
                        if nbr_zeros > 0:
                            lignebloc += "#" + str(nbr_zeros) + " "
                            nbr_zeros = 0
                        lignebloc += str(int(blocs_compresses[i][l][m][1])) + " "
                else:
                   lignebloc += str(int(blocs_compresses[i][l][m][1])) + " "
        # Ajouter le nombre de zéros à la fin du bloc.
        if nbr_zeros > 0:
            lignebloc += "#" + str(nbr_zeros) + " "
        text += lignebloc.strip() + "\n"
        # Boucle Cr.
        lignebloc = ""
        nbr_zeros = 0
        for o in range(len(blocs_compresses[i])):
            for n in range(len(blocs_compresses[i][j])):
                if modecomp:
                    if int(blocs_compresses[i][o][n][2]) == 0:
                        nbr_zeros += 1
                    else:
                        if nbr_zeros > 0:
                            lignebloc += "#" + str(nbr_zeros) + " "
                            nbr_zeros = 0
                        lignebloc += str(int(blocs_compresses[i][o][n][2])) + " "
                else:
                   lignebloc += str(int(blocs_compresses[i][o][n][2])) + " "
        # Ajouter le nombre de zéros à la fin du bloc.
        if nbr_zeros > 0:
            lignebloc += "#" + str(nbr_zeros) + " "
        text += lignebloc.strip() + "\n"
    return text  # On retourne le texte entier.

## Décompression

In [ ]:
def decompression_image(imagepad,h,l):
    """Fonction de decompression qui prend une liste de blocs pour chaque canal Y,Cb,Cr contenant les coefficients de la DCT et calcule une matrice représentant l'image en RGB"""
    global image  # On définit en variable global l'image de base afin de retrouver ses dimensions sans le padding.
    blocs_8x8 = de_cos_discret(imagepad)  # On repasse l'image en blocs de 8x8 sans fréquences.
    image_ycbcr = regrouper_blocs(blocs_8x8,h,l)  # On regroupe l'image, en retirant les blocs de 8x8.
    image_rgb = rgb(image_ycbcr)  # On repasse l'image en RGB.
    image_sans_padding = depadding(image_rgb,image.shape[0],image.shape[1])  # On retire le padding de notre image. En utilisant les dimensions de l'image de base sans padding.
    return image_sans_padding  # On renvoie l'image en RGB sans le padding.


def decompression(ligne):
    """Fonction qui prend une ligne d'un fichier SPJG et la décompresse"""
    liste=[]  # On définit une liste vide.
    ligne = ligne.split()  # Transforme notre ligne liste de liste contenant chaque éléments.
    for i in ligne:  # On parcourt chaque éléments de notre liste.
        mot = i  # On définit mot égal au mot actuel.
        if not "#" in mot:  # Si # n'est pas dans le mot.
            liste.append(int(mot))  # On ajoute à notre liste final l'élément séléctionné auparavant.
        else:
            mot = mot[1:]  # On sélectionne l'élément sans le #, ex: #64 = 64.
            nbr_zeros = int(mot)  # On définit la variable nombre zéros qui correspond au nombre de 0 dans le #.
            for j in range(nbr_zeros):  # On boucle le nombre de 0.
                liste.append(0)  # On ajoute chaque 0 à notre liste final.
    return liste  # On return notre liste qui correspond à notre ligne sans la compression RLE.


def conversion(liste_decompressee):
    """Fonction qui convertit une ligne de 64 éléments en blocs de 8x8"""
    mat=np.zeros((8,8), dtype=int)  # On crée une matrice de zeros en 8x8 pour y mettre les valeurs. 
    nbr=0  # On crée une variable pour parcourir toute la liste des valeurs normalement.
    for i in range(8):  # La boucle pour parcourir le bloc de 8x8.
        for j in range(8):
            mat[i][j] = liste_decompressee[nbr]  # On ajoute les valeurs de la liste au bloc de 8x8.
            nbr+=1  # On augmente la variable pour parcourir la liste.
    return mat  # On renvoi le bloc de 8x8 avec les valeurs.


def lecture():
    """Fonction qui lit un fichier SJPG et qui cree les listes de blocs decrites par le fichier"""
    texte = open("fichier SJPG","r")
    for k in range(4):  # On retire les 4 premières lignes de notre ficher, qui correspondent aux infos de l'image.
        texte.readline()
    lignes=[]  # On créer une liste vide.
    for i in texte:  # On parcourt chaque ligne de notre texte.
        lignes.append(i)  # On les ajoute à notre liste lignes.
    blocs=[]  # On créer une liste vide.
    for i in lignes:  # Pour chaque éléments dans lignes.
        ligne_64 = decompression(i)  # On décompresse chaque ligne de notre liste lignes.
        bloc_8x8 = conversion(ligne_64)  # On convertit cette ligne en bloc de 8x8.
        blocs.append(bloc_8x8)  # On ajoute ce bloc à notre liste blocs.
    return np.array(blocs)  # On return les blocs obtenu à partir de notre décompression.

## Tests 

In [ ]:
"""Chargement de l'image de base en RGB"""
image = load("test.png")
#Affichage de l'image
Image.fromarray(image,mode ='RGB').show()


"""Image transformée en YCbCr"""
mat_ycbcr = ycbcr(image)
#Affichage de l'image
Image.fromarray(mat_ycbcr,mode ='YCbCr').show()


"""Image retransformée en RGB"""
mat_rgb = rgb(mat_ycbcr)
#Affichage de l'image
Image.fromarray(mat_rgb,mode ='RGB').show()


"""Image avec le padding"""
mat_pad = padding(mat_rgb)
#Affichage de l'image
Image.fromarray(mat_pad,mode ='RGB').show()
mat_pad_ycbcr = ycbcr(mat_pad)
#Affichage de l'image
Image.fromarray(mat_pad_ycbcr,mode ='YCbCr').show()


"""Image sans padding"""
mat_sans_padding = depadding(mat_rgb,image.shape[0],image.shape[1])
#Affichage de l'image
Image.fromarray(mat_sans_padding,mode ='RGB').show()


"""Sous echantillonnage de l'image"""
sous_echantillon = sous_echantillonnage(mat_pad_ycbcr)


"""Dé échantillonnage de l'image"""
de_echantillon = de_echantillonnage(sous_echantillon)
#Affichage de l'image
Image.fromarray(de_echantillon,mode ='YCbCr').show()


"""Découpage en blocs de l'image"""
blocs_decoupe = decouper_en_blocs(de_echantillon)


"""Regroupe l'image découpée en blocs"""
blocs_regroupe = regrouper_blocs(blocs_decoupe,mat_pad.shape[0],mat_pad.shape[1])
#Affichage de l'image
Image.fromarray(blocs_regroupe,mode ='YCbCr').show()


"""Transforme en fréquence les blocs découpée"""
freq = cos_discret(blocs_decoupe)


"""Affichage de l'image après avoir retiré les fréquences"""
defreq = de_cos_discret(freq)
#Affichage de l'image
Image.fromarray(regrouper_blocs(defreq,mat_pad.shape[0],mat_pad.shape[1]),mode ='YCbCr').show()


"""Test du mode 0"""
mode0_resultat = mode0(blocs_decoupe)
#Affichage de l'image
Image.fromarray(depadding(rgb(regrouper_blocs(de_cos_discret(mode0_resultat),mat_pad_ycbcr.shape[0],mat_pad_ycbcr.shape[1])),image.shape[0],image.shape[1]),mode = "RGB").show()


"""Seuil pour test"""
seuildonne = 100

"""Test du mode 1, avec un seuil"""
mode1_resultat = mode1(blocs_decoupe,seuildonne)
#Affichage de l'image
Image.fromarray(depadding(rgb(regrouper_blocs(de_cos_discret(mode1_resultat),mat_pad_ycbcr.shape[0],mat_pad_ycbcr.shape[1])),image.shape[0],image.shape[1]),mode = "RGB").show()


"""Test du mode 2, avec un seuil"""
mode2_resultat = mode2(blocs_decoupe,seuildonne)
#Affichage de l'image
Image.fromarray(depadding(rgb(regrouper_blocs(de_cos_discret(mode2_resultat),mat_pad_ycbcr.shape[0],mat_pad_ycbcr.shape[1])),image.shape[0],image.shape[1]),mode = "RGB").show()


"""Fonction qui a partir d'une image RGB, créer les listes de blocs compressés dans les 3 modes"""
explosion(image,seuildonne)


"""Fonction écriture qui prend une image en YCbCr et le met en int"""
message_ecriture = ecriture(mat_ycbcr,1)


"""Fonction de decompression qui prend une liste de blocs pour chaque canal Y,Cb,Cr contenant les coefficients de la DCT et calcule une matrice représentant l'image en RGB."""
l_blocs_YCbCr_DCT = cos_discret(decouper_en_blocs(ycbcr(padding(image))))  # Liste blocs en YCbCr transformé en fréquence.
image_decompressee = decompression_image(l_blocs_YCbCr_DCT,padding(image).shape[0],padding(image).shape[1])  # padding(image).shape correspond à la hauteur et la largeur de l'image avec le padding.
Image.fromarray((image_decompressee),mode ='RGB').show()


"""Fonction qui lit un fichier SJPG et qui créé les listes de blocs décrites par le fichier"""
print(lecture())